In [0]:
%run ./cid_mapping_common_business

#1.1 batch recode to vertex

In [0]:
# ==============================
# clean 数据（source 层）
# ==============================
def get_batch_data(task_id):

    # batch 筛选特定market 和rakuten、linefift来源数据
    t_clean_consumer = (
        get_RL_clear_consumer_df(task_id)
        .withColumn("record_type", F.lit(CID_MATCH_RECORD_TYPE_BATCH))
        .withColumn("master_recode_create_time", F.lit(None).cast(TimestampType()))
    )

    clean_phone_vld = (
        spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_phone")
        .filter(F.col("task_id") == task_id)
        .filter(F.col("srcp_mrkt_code").isin(MARKETS_ENABLE_CID_MATCH))
    )

    clean_emedia_vld = (
        spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_emedia")
        .filter(F.col("task_id") == task_id)
        .filter(F.col("srce_mrkt_code").isin(MARKETS_ENABLE_CID_MATCH))
        .withColumn("srce_address", F.lower(F.col("srce_address")))
    )


    # ==============================
    # 1.1 构建 batch profile（batch_df）
    # ==============================
    batch_df = (
        t_clean_consumer.alias("a")
        .join(clean_phone_vld.alias("b"),
            (F.col("b.SRCP_SRCC_ID") == F.col("a.SRCC_ID")) &
            (F.col("b.srcp_mrkt_code") == F.col("a.srcc_mrkt_code")),
            "left")
        .join(clean_emedia_vld.alias("c"),
            (F.col("c.SRCE_SRCC_ID") == F.col("a.SRCC_ID")) &
            (F.col("c.srce_mrkt_code") == F.col("a.srcc_mrkt_code")),
            "left")
        .select(
            F.lit(None).cast("string").alias("scon_id"),
            F.lit(None).cast("string").alias("mapping_conusmer_id"),

            F.col("a.srcc_id").alias("scon_srcc_id"),
            F.col("a.srcc_srcs_code").alias("scon_srcs_code"),
            F.col("a.srcc_sourcetimestamp").alias("scon_sourcetimestamp"),
            F.col("a.srcc_mrkt_code").alias("scon_mrkt_code"),
            F.col("a.srcc_brnd_code").alias("scon_brnd_code"),
            F.col("a.srcc_consumerid").alias("order_id"),

            F.col("a.srcc_englishfirstname").alias("scon_englishfirstname"),
            F.col("a.srcc_englishmiddlename").alias("scon_englishmiddlename"),
            F.col("a.srcc_englishlastname").alias("scon_englishlastname"),
            F.col("a.srcc_englishfullname").alias("scon_englishfullname"),

            F.col("a.srcc_localfirstname").alias("scon_localfirstname"),
            F.col("a.srcc_localmiddlename").alias("scon_localmiddlename"),
            F.col("a.srcc_locallastname").alias("scon_locallastname"),
            F.col("a.srcc_localfullname").alias("scon_localfullname"),

            F.col("a.srcc_localfirstname2").alias("scon_localfirstname2"),
            F.col("a.srcc_localmiddlename2").alias("scon_localmiddlename2"),
            F.col("a.srcc_locallastname2").alias("scon_locallastname2"),
            F.col("a.srcc_localfullname2").alias("scon_localfullname2"),

            # F.coalesce(F.col("c.srce_address"), F.lit("")).alias("scme_address"),
            # F.coalesce(F.col("b.srcp_phonenumber"), F.lit("")).alias("scph_phonenumber"),

            F.when(F.col("c.SRCE_SRCC_ID").isNull(), F.lit(""))
                .when((F.col("c.srce_quality_code") == "vld") | (F.col("c.srce_mrkt_code").isin(MARKETS_ENABLE_LINE_MEDIA) & (F.col("c.srce_emdt_code") == "scllneprs")), 
                      F.col("c.srce_address"))
                .otherwise(F.lit(None)).alias("scme_address"),

            F.when(F.col("b.SRCP_SRCC_ID").isNull(), F.lit(""))
                .when(F.col("b.srcp_quality_code") == "vld", F.col("b.srcp_phonenumber"))
                .otherwise(F.lit(None)).alias("scph_phonenumber"),

            F.col("a.record_type"),
            F.col("a.master_recode_create_time"),
            F.col("a.batch_id")
        ).distinct()
    )


    return t_clean_consumer, batch_df

#1.2 master recode to vertex

In [0]:
def is_scon_name_empty(tab_alias: str) -> Column:
    """判断 master 姓名字段是否全空（8个字段）"""
    cols = [
        f"{tab_alias}.scon_englishfirstname", f"{tab_alias}.scon_englishmiddlename",
        f"{tab_alias}.scon_englishlastname",  f"{tab_alias}.scon_englishfullname",
        f"{tab_alias}.scon_localfirstname",   f"{tab_alias}.scon_localmiddlename",
        f"{tab_alias}.scon_locallastname",    f"{tab_alias}.scon_localfullname"
    ]
    cond = F.lit(True)
    for c in cols:
        cond = cond & (F.col(c).isNull() | (F.trim(F.col(c)) == ""))
    return cond

In [0]:
def get_path1_data(t_master_consumer, t_master_phone_vld, t_master_emedia_vld, batch_df):
    path1 = (
        t_master_consumer.alias("con")
        .filter((~is_scon_name_empty("con")))
        .join(t_master_phone_vld.alias("ph"),
            (F.col("con.scon_id") == F.col("ph.scph_scon_id")) &
            (F.col("con.scon_mrkt_code") == F.col("ph.scph_mrkt_code")),
            "left")
        .join(t_master_emedia_vld.alias("me"),
            (F.col("con.scon_id") == F.col("me.scme_scon_id")) &
            (F.col("con.scon_mrkt_code") == F.col("me.scme_mrkt_code")),
            "left")
        .select(
            F.col("con.scon_id"),
            F.col("con.scon_consumerid").alias("mapping_conusmer_id"),
            
            F.col("con.scon_srcc_id"),
            F.col("con.scon_srcs_code"),
            F.col("con.scon_sourcetimestamp"),
            F.col("con.scon_mrkt_code"),
            F.col("con.scon_brnd_code"),
            F.lit(None).cast("string").alias("order_id"),

            F.col("con.scon_englishfirstname"),
            F.col("con.scon_englishmiddlename"),
            F.col("con.scon_englishlastname"),
            F.col("con.scon_englishfullname"),
            F.col("con.scon_localfirstname"),
            F.col("con.scon_localmiddlename"),
            F.col("con.scon_locallastname"),
            F.col("con.scon_localfullname"),
            F.col("con.scon_localfirstname2"),
            F.col("con.scon_localmiddlename2"),
            F.col("con.scon_locallastname2"),
            F.col("con.scon_localfullname2"),
            # F.coalesce(F.col("me.scme_address"), F.lit("")).alias("scme_address"),
            # F.coalesce(F.col("ph.scph_phonenumber"), F.lit("")).alias("scph_phonenumber"),

            F.when(F.col("me.scme_scon_id").isNull(), F.lit(""))
                .when((F.col("me.scme_quality_code") == "vld") | (F.col("me.scme_mrkt_code").isin(MARKETS_ENABLE_LINE_MEDIA) & (F.col("me.scme_emdt_code") == "scllneprs")), 
                      F.col("me.scme_address"))
                .otherwise(F.lit(None)).alias("scme_address"),

            F.when(F.col("ph.scph_scon_id").isNull(), F.lit(""))
                .when(F.col("ph.scph_quality_code") == "vld", F.col("ph.scph_phonenumber"))
                .otherwise(F.lit(None)).alias("scph_phonenumber"),
            F.col("con.record_type"),
            F.col("con.master_recode_create_time"),
            F.col("con.batch_id")
        )
        .filter( ~((F.col("scme_address") ==  F.lit("")) & (F.col("scph_phonenumber") == F.lit(""))) ).alias("master_tab")
        .join(batch_df.select("scon_mrkt_code", "scme_address", "scph_phonenumber").distinct().alias("batch_tab"),
            (F.col("master_tab.scon_mrkt_code") == F.col("batch_tab.scon_mrkt_code")) & 
            (F.col("master_tab.scme_address") == F.col("batch_tab.scme_address")) &
            (F.col("master_tab.scph_phonenumber") == F.col("batch_tab.scph_phonenumber")),
            "inner"
        )
        .select(
            "master_tab.*"
        )
        .distinct()
    )

    return path1


def get_path2_data(t_master_consumer, t_master_phone_vld, t_master_emedia_vld, t_clean_consumer):
    # 同一order id,关联相同mapping_cid
    t_transaction_master = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_transaction_master")

    path2 = (
        t_clean_consumer.alias("tcc_tab")
        .join(t_transaction_master.alias("ttm_tab"), 
            (F.col("tcc_tab.srcc_mrkt_code") == F.col("ttm_tab.tran_mrkt_code")) &
            (F.col("tcc_tab.srcc_brnd_code") == F.col("ttm_tab.tran_brnd_code")) &
            (F.col("tcc_tab.srcc_srcs_code") == F.col("ttm_tab.tran_srcs_code")) &
            (F.col("tcc_tab.srcc_consumerid") == F.col("ttm_tab.tran_order_id")),
            "inner"
        )
        .join(
            t_master_consumer.alias("con"),
            (F.col("tcc_tab.srcc_mrkt_code") == F.col("con.scon_mrkt_code")) &
            (F.col("tcc_tab.srcc_brnd_code") == F.col("con.scon_brnd_code")) &
            (F.col("tcc_tab.srcc_srcs_code") == F.col("con.scon_srcs_code")) &
            (F.col("ttm_tab.tran_mapping_consumer_id") == F.col("con.scon_consumerid")),
            "inner"
        )
        .join(t_master_phone_vld.alias("ph"),
            (F.col("con.scon_id") == F.col("ph.scph_scon_id")) &
            (F.col("con.scon_mrkt_code") == F.col("ph.scph_mrkt_code")),
            "left")
        .join(t_master_emedia_vld.alias("me"),
            (F.col("con.scon_id") == F.col("me.scme_scon_id")) &
            (F.col("con.scon_mrkt_code") == F.col("me.scme_mrkt_code")),
            "left")
        .select(
            F.col("con.scon_id"),
            F.col("con.scon_consumerid").alias("mapping_conusmer_id"),
            
            F.col("con.scon_srcc_id"),
            F.col("con.scon_srcs_code"),
            F.col("con.scon_sourcetimestamp"),
            F.col("con.scon_mrkt_code"),
            F.col("con.scon_brnd_code"),
            F.col("tcc_tab.srcc_consumerid").alias("order_id"),

            F.col("con.scon_englishfirstname"),
            F.col("con.scon_englishmiddlename"),
            F.col("con.scon_englishlastname"),
            F.col("con.scon_englishfullname"),
            F.col("con.scon_localfirstname"),
            F.col("con.scon_localmiddlename"),
            F.col("con.scon_locallastname"),
            F.col("con.scon_localfullname"),
            F.col("con.scon_localfirstname2"),
            F.col("con.scon_localmiddlename2"),
            F.col("con.scon_locallastname2"),
            F.col("con.scon_localfullname2"),
            # F.coalesce(F.col("me.scme_address"), F.lit("")).alias("scme_address"),
            # F.coalesce(F.col("ph.scph_phonenumber"), F.lit("")).alias("scph_phonenumber"),
            F.when(F.col("me.scme_scon_id").isNull(), F.lit(""))
                .when((F.col("me.scme_quality_code") == "vld") | (F.col("me.scme_mrkt_code").isin(MARKETS_ENABLE_LINE_MEDIA) & (F.col("me.scme_emdt_code") == "scllneprs")), 
                      F.col("me.scme_address"))
                .otherwise(F.lit(None)).alias("scme_address"),

            F.when(F.col("ph.scph_scon_id").isNull(), F.lit(""))
                .when(F.col("ph.scph_quality_code") == "vld", F.col("ph.scph_phonenumber"))
                .otherwise(F.lit(None)).alias("scph_phonenumber"),
            F.col("con.record_type"),
            F.col("con.master_recode_create_time"),
            F.col("con.batch_id")
        )
    )

    return path2



In [0]:
def get_master_data(t_clean_consumer, batch_df):

    t_master_consumer = (
        get_RL_master_consumer_df()
        .withColumn("record_type", F.lit(CID_MATCH_RECORD_TYPE_CM))
        .withColumn("master_recode_create_time", F.col("scon_creation_dt"))
    )

    t_master_phone_vld = (
        spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_phone")
        .filter(F.col("scph_mrkt_code").isin(MARKETS_ENABLE_CID_MATCH))
        # .filter(F.col("scph_quality_code") == "vld")
        # .filter(F.coalesce(F.col("scph_phonenumber"), F.lit("")) != "")
    )

    # (JPN LINE 媒体扩展) 
    t_master_emedia_vld = (
        spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_emedia")
        .filter(F.col("scme_mrkt_code").isin(MARKETS_ENABLE_CID_MATCH))
        # .filter(
        #     (F.col("scme_quality_code") == "vld") |
        #     (F.col("scme_mrkt_code").isin(MARKETS_ENABLE_LINE_MEDIA) & (F.col("scme_emdt_code") == "scllneprs"))
        # )
        .withColumn("scme_address", F.lower(F.col("scme_address")))
    )


    path1_df = get_path1_data(t_master_consumer, t_master_phone_vld, t_master_emedia_vld, batch_df)
    path2_df = get_path2_data(t_master_consumer, t_master_phone_vld, t_master_emedia_vld, t_clean_consumer)

    master_df = (
        path1_df
        .unionByName(path2_df)
        .distinct()
    )

    return master_df

#1.3 all record with match key

In [0]:
def generate_match_key(batch_df, master_df):
    merged_df = (
        batch_df.unionByName(master_df)
        .withColumn("EmailMatchKey", cid_email_match_udf(F.col("scme_address")))
        .withColumn("PhoneMatchKey", cid_phone_match_udf(F.col("scph_phonenumber")))

        .withColumn("EnglishLNameMatchKey", cleanse_name_udf(F.col("scon_englishlastname")))
        .withColumn("EnglishFNameMatchKey", cleanse_name_udf(F.col("scon_englishfirstname")))
        .withColumn("LocalLNameMatchKey", cleanse_name_udf(F.col("scon_locallastname")))
        .withColumn("LocalFNameMatchKey", cleanse_name_udf(F.col("scon_localfirstname")))
        .withColumn("LocalLNameMatchKey2", cleanse_name_udf(F.col("scon_locallastname2"))) 
        .withColumn("LocalFNameMatchKey2", cleanse_name_udf(F.col("scon_localfirstname2")))

        .withColumn("name_key", 
            F.when(F.col("scon_mrkt_code").isin(MARKETS_ENABLE_LOCALNAME2), 
                full_name_sort_match_key_v3_udf(F.col("scon_englishfirstname"), F.col("scon_englishlastname"), F.col("scon_englishfullname"),
                                                F.col("scon_localfirstname"), F.col("scon_locallastname"), F.col("scon_localfullname"),
                                                F.col("scon_localfirstname2"), F.col("scon_locallastname2"), F.col("scon_localfullname2")))
            .otherwise(full_name_sort_match_key_v2_udf(F.col("scon_englishfirstname"), F.col("scon_englishlastname"), F.col("scon_englishfullname"),
                                                    F.col("scon_localfirstname"), F.col("scon_locallastname"), F.col("scon_localfullname"))))
        .withColumn("FullNameMatchKey", F.col("name_key.asc"))
        .withColumn("FullNameMatchKeyDesc", F.col("name_key.desc"))

        ).drop("name_key", "scad_city_localdesc", "scad_postalcode")
    
    return merged_df

#2.1 vertices, edge generate

In [0]:
def generate_vertices_and_edge(df_with_id):

    _mk = (F.col("a.scon_mrkt_code") == F.col("b.scon_mrkt_code"))

    # 规则 A: Email + Phone + Name
    condA_base =  _mk & (F.col("a.EmailMatchKey") == F.col("b.EmailMatchKey")) & (F.col("a.PhoneMatchKey") == F.col("b.PhoneMatchKey")) & (F.concat(F.col("a.EmailMatchKey"), F.col("b.EmailMatchKey"), F.col("a.PhoneMatchKey"), F.col("b.PhoneMatchKey")) != F.lit(""))

    # EnglishFName 通过 编辑距离算法 计算文本相似度, 相似度>=0.85 则通过match
    common_english_name_cond = ( 
        (F.col("a.EnglishLNameMatchKey") == F.col("b.EnglishLNameMatchKey")) & 
        
        (F.when(F.greatest(F.length("a.EnglishFNameMatchKey"), F.length("b.EnglishFNameMatchKey")) == F.lit(0), F.lit(1.0)) 
            .otherwise( F.lit(1) - (F.levenshtein(F.col("a.EnglishFNameMatchKey"), F.col("b.EnglishFNameMatchKey")) / F.greatest(F.length("a.EnglishFNameMatchKey"), F.length("b.EnglishFNameMatchKey"))))  >= 0.85
        ) &

        (F.concat(F.col("a.EnglishLNameMatchKey"), F.col("b.EnglishLNameMatchKey"), F.col("a.EnglishFNameMatchKey"), F.col("b.EnglishFNameMatchKey")) != F.lit("")) 
    )
    # (F.col("a.EnglishFNameMatchKey") == F.col("b.EnglishFNameMatchKey")) & 

    common_local_name_cond = (
        (F.col("a.LocalFNameMatchKey") == F.col("b.LocalFNameMatchKey")) & (F.col("a.LocalLNameMatchKey") == F.col("b.LocalLNameMatchKey")) &
        (F.concat(F.col("a.LocalFNameMatchKey"), F.col("b.LocalFNameMatchKey"), F.col("a.LocalLNameMatchKey"), F.col("b.LocalLNameMatchKey")) != F.lit(""))
    )

    common_local_name2_cond = (
        (F.col("a.LocalFNameMatchKey2") == F.col("b.LocalFNameMatchKey2")) & (F.col("a.LocalLNameMatchKey2") == F.col("b.LocalLNameMatchKey2")) &
        (F.concat(F.col("a.LocalFNameMatchKey2"), F.col("b.LocalFNameMatchKey2"), F.col("a.LocalLNameMatchKey2"), F.col("b.LocalLNameMatchKey2")) != F.lit(""))
    )

    condA1 = condA_base & (F.col("a.FullNameMatchKey") == F.col("b.FullNameMatchKey"))

    # EnglishFName 通过 编辑距离算法 计算文本相似度, 相似度>=0.85 则通过match
    condA2 = condA_base & common_english_name_cond

    condA3 =  condA_base & common_local_name_cond
    condA4 =  condA_base & common_local_name2_cond
    condA5 = condA_base & (F.col("a.FullNameMatchKeyDesc") == F.col("b.FullNameMatchKeyDesc"))

    # 规则 D: order_id
    condD = _mk & (F.col("a.scon_brnd_code") == F.col("b.scon_brnd_code")) & (F.col("a.scon_srcs_code") == F.col("b.scon_srcs_code")) & (F.col("a.order_id") == F.col("b.order_id")) 

    # 规则 E: mapping_conusmer_id
    condE = _mk & (
        (F.col("a.record_type") == F.lit(CID_MATCH_RECORD_TYPE_CM)) & (F.col("b.record_type") == F.lit(CID_MATCH_RECORD_TYPE_CM)) &
        (F.col("a.mapping_conusmer_id") == F.col("b.mapping_conusmer_id")) &
        (F.col("a.mapping_conusmer_id").isNotNull())
    )

    all_match_conditions = (condA1)|(condA2)|(condA3)|(condA4)|(condA5) | (condD) | (condE)

    full_cond = (all_match_conditions) & (F.col("a.id") < F.col("b.id"))

    match_types_col = F.array_compact(F.array(
        F.when(condA1, F.lit("FullName_Email_Phone_Match")),        # FullNameMatchKey + Email + Phone
        F.when(condA2, F.lit("EnglishName_Email_Phone_Match")),     # EnglishLName + EnglishFName + Email + Phone
        F.when(condA3, F.lit("LocalName_Email_Phone_Match")),       # LocalFName + LocalLName + Email + Phone
        F.when(condA4, F.lit("LocalName2_Email_Phone_Match")),      # LocalFName2 + LocalLName2 + Email + Phone
        F.when(condA5, F.lit("FullNameDesc_Email_Phone_Match")),    # FullNameMatchKeyDesc + Email + Phone
        F.when(condD,  F.lit("OrderId_Match")),                     # OrderId + brnd_code + srcs_code
        F.when(condE,  F.lit("MappingConusmerId_Match")) 
    ))


    vertices = df_with_id.select(F.col("id")).distinct()

    edges = (df_with_id.alias("a")
            .join(df_with_id.alias("b"), full_cond, "inner")
            .select(
                F.col("a.id").alias("src"),
                F.col("b.id").alias("dst"),

                F.col("a.scon_mrkt_code").alias("src_mrkt_code"),
                F.col("a.scon_srcc_id").alias("src_srcc_id"),
                
                F.col("b.scon_mrkt_code").alias("dst_mrkt_code"),
                F.col("b.scon_srcc_id").alias("dst_srcc_id"),
                
                match_types_col.alias("match_types")
            )
            .groupBy("src", "dst", "src_mrkt_code", "src_srcc_id", "dst_mrkt_code", "dst_srcc_id")
            .agg(
                F.array_join(
                    F.array_distinct(
                        F.flatten(F.collect_list("match_types"))
                    ), ","
                ).alias("match_types")
            )
    )

    return vertices, edges

#2.2 graph generate

In [0]:

def generate_gid(vertices, edges, df_with_id):

    g = GraphFrame(vertices, edges)
    spark.sparkContext.setCheckpointDir(f"{get_env_config('checkpoint_path_consumer')}/{task_id}/cid_match")
    print(f"tCheckpointDir: {spark.sparkContext.getCheckpointDir()}")
    
    components = g.connectedComponents(algorithm="graphx", checkpointInterval=2)

    target_df = df_with_id.join(
        components,
        on="id", how="left"
    )

    grp_size_df = target_df.groupBy("component").agg(F.countDistinct("scon_srcc_id").alias("grp_size"))

    final_df = target_df.join(grp_size_df, on="component", how="left").select(
        F.col("scon_srcc_id").alias("srcc_id"),
        F.col("scon_mrkt_code").alias("mrkt_code"),
        F.col("scon_brnd_code").alias("brnd_code"),
        F.col("scon_srcs_code").alias("source_code"),

        F.col("order_id"),
        F.col("scon_sourcetimestamp").alias("source_timestamp"),
        F.col("scon_id").alias("master_scon_id"),

        F.col("mapping_conusmer_id"),
        F.col("master_recode_create_time"),
        F.col("record_type"),
        F.col("component").alias("gid"),
        F.col("grp_size"),
        F.col("batch_id")
    ) \
    .distinct() \
    .withColumn("matc_id", F.expr("uuid()")) \
    .withColumn("task_id", F.lit(task_id)) \
    .withColumn("match_type", F.lit(MATCH_TYPE_REGULAR_STR)) \
    .withColumn("creation_dt", F.current_timestamp()) 

    return final_df

#3 mapping cid generate

In [0]:
def generate_new_cid(final_df):
    #  is_history_flag 判断条件是为了兼容history数据 与 新数据 中scon_id生成逻辑不同导致的排序异常, 保证history数据 排序优先级高于 新数据
    #  history数据 scon_id 为单调递增数值, 因此scon_id越小代表数据越早
    #  新数据 scon_id 为uuid, 因此需要通过 recode_create_time 判断最早数据
    earliest_master_cid = (
        final_df
        .filter(F.col("record_type") == F.lit(CID_MATCH_RECORD_TYPE_CM))
        .withColumn("is_history_flag", F.when(F.length(F.col("master_scon_id")) < F.lit(36), F.lit(1)).otherwise(F.lit(0)))
        .withColumn("master_recode_create_time", F.when(F.col("is_history_flag") == F.lit(1), F.to_timestamp(F.lit("1900-01-01"))).otherwise(F.col("master_recode_create_time")))
        .withColumn("earliest_rank", F.row_number().over(
            Window.partitionBy("gid").orderBy(
                F.col("is_history_flag").desc(),
                F.col("master_recode_create_time").asc(),
                F.col("master_scon_id").asc()
            )
        ))
        .filter(F.col("earliest_rank") == 1)
        .select(F.col("gid"), F.col("mapping_conusmer_id").alias("new_mapping_conusmer_id"))
    )

    group_cid = (
        final_df.select("gid").distinct()
        .join(earliest_master_cid, ["gid"], "left")
        .withColumn("new_mapping_conusmer_id",
            F.when(F.col("new_mapping_conusmer_id").isNull(), F.expr("uuid()"))
            .otherwise(F.col("new_mapping_conusmer_id")))
    )

    final_with_newCid_df = final_df.join(group_cid, ["gid"], "left")

    return final_with_newCid_df

#4 check cid duplicate

In [0]:
def check_cid_duplicate(task_id, logger):
    # 1. check
    cid_with_flag_df = get_cidGroup_with_flag(task_id)

    # 2. modify log content
    masterCid_duplicate_records = [
        row[0] 
        for row in cid_with_flag_df
            .filter(F.col("record_type") == CID_MATCH_RECORD_TYPE_CM)
            .filter(F.col("is_masterCid_duplicate") == True)
            .select("matc_id")
            .collect()
    ]

    if len(masterCid_duplicate_records) > 0:
        masterCid_duplicate_records_str = ",".join([f"'{item}'" for item in masterCid_duplicate_records])
        logger.status = "WARNING"
        logger.message = f"The following records have multiple new_mapping_conusmer_id corresponding to the same mapping_conusmer_id, matc_id is: {masterCid_duplicate_records_str}"



    order_duplicate_records = [
        row[0]
        for row in cid_with_flag_df
            .filter(F.col("is_order_duplicate") == True)
            .select("matc_id")
            .collect()
    ]

    if len(order_duplicate_records) > 0:
        order_duplicate_records_str = ",".join([f"'{item}'" for item in order_duplicate_records])
        if logger.status == "WARNING":
            logger.message =  f"{logger.message} \nThe following records have multiple new_mapping_conusmer_id corresponding to the same mrkt_comde, brnd_comde, source_comde, and order_id, matc_id is: {order_duplicate_records_str}"
        else:
            logger.status = "WARNING"
            logger.message = f"The following records have multiple new_mapping_conusmer_id corresponding to the same mrkt_comde, brnd_comde, source_comde, and order_id, matc_id is: {order_duplicate_records_str}"
     

    # 3. mark exclude data
    update_exclude_df = (cid_with_flag_df
            .filter(F.col("record_type") == CID_MATCH_RECORD_TYPE_BATCH)
            .filter(F.col("is_order_duplicate") == True)
            .select("task_id", "mrkt_code", "srcc_id")
            .distinct()
    )

    if update_exclude_df.count() > 0:
        clean_consumer_delta_table = DeltaTable.forName(spark, f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_consumer")
        
        (clean_consumer_delta_table
            .alias("target")
            .merge(
                update_exclude_df.alias("source"),
                "target.TASK_ID = source.task_id AND "
                "target.SRCC_MRKT_CODE = source.mrkt_code AND "
                "target.SRCC_ID = source.srcc_id "
            )
            .whenMatchedUpdate(
                set={
                    "IS_INCLUDE": F.lit(False),
                    "EXCLUDE_TYPE": F.lit(EXCLUDE_TYPE_BY_CID_DUPLICATE)
                }
            )
            .execute()
        )



#main

In [0]:
def cid_match_process(task_id, logger):

    # 1.1 batch data
    t_clean_consumer, batch_df = get_batch_data(task_id)
    
    # 1.2 master data
    master_df = get_master_data(t_clean_consumer, batch_df)

    # 1.3 batch union master, generate match key
    merged_df = generate_match_key(batch_df, master_df)

    # 2.1 generate vertices, edges
    df_with_id = merged_df.withColumn("id", F.concat_ws("_", F.col("scon_mrkt_code"), F.col("scon_srcc_id")))
    vertices, edges = generate_vertices_and_edge(df_with_id)

    df_with_id.cache()
    edges.cache()

    save_to_target_table(
        edges.withColumn("task_id", F.lit(task_id)).withColumn("creation_dt", F.current_timestamp()),
        f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_cid_edge",
        f"task_id = '{task_id}'"
    )

    # 2.2 generate gid
    final_df = generate_gid(vertices, edges, df_with_id)

    # 3. generate new cid 
    final_with_newCid_df = generate_new_cid(final_df)

    save_to_target_table(
        final_with_newCid_df,
        f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_cid_group",
        f"task_id = '{task_id}' and match_type = '{MATCH_TYPE_REGULAR_STR}' "
    )
    

    df_with_id.unpersist()
    edges.unpersist()

    # 4. check cid duplicate
    check_cid_duplicate(task_id, logger)

In [0]:
task_id = dbutils.widgets.get("task_id")
print(f"task_id: {task_id}")

with StepLogger("3.1_cid_match", "03-1", "consumerlist", task_id=task_id) as logger:
    cid_match_process(task_id, logger)